In [1]:
!pip install torch-pruning scipy tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torchvision.models import vgg16
from tqdm import tqdm
import numpy as np
from scipy.stats import spearmanr
import torch_pruning as tp
import copy
from sklearn.metrics import precision_recall_fscore_support

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 128
PRUNE_RATIO = 0.2        # 🔥 SAFE
EPOCHS_FINETUNE = 100

CALIB_SAMPLES = 2000

LR = 0.01                # 🔥 LOWER LR
WEIGHT_DECAY = 5e-4

print("Device:", DEVICE)

Device: cuda


In [4]:
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),
                (0.2470,0.2435,0.2616))
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914,0.4822,0.4465),
                (0.2470,0.2435,0.2616))
])

train_set = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_tf
)

test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_tf
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 170M/170M [00:05<00:00, 31.2MB/s]


In [5]:
model = vgg16(num_classes=10)

model.features[0] = nn.Conv2d(3, 64, 3, 1, 1)
model.features[4] = nn.Identity()

model.avgpool = nn.AdaptiveAvgPool2d((1,1))

model.classifier = nn.Sequential(
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 100)
)

# Disable inplace ReLU (CRITICAL)
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False

model = model.to(DEVICE)

In [6]:
PATH = "/kaggle/input/datasets/rahulboddeda/vgg16-cifar10-baseline/vgg16_baseline_best_CIFAR10.pt"

model.load_state_dict(torch.load(PATH, map_location=DEVICE))

print("✅ Pretrained model loaded")

✅ Pretrained model loaded


In [7]:
def evaluate(model):
    model.eval()
    correct,total = 0,0
    with torch.no_grad():
        for x,y in test_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)
    return correct/total

In [8]:
idx = torch.randperm(len(train_set))[:CALIB_SAMPLES]
calib_loader = DataLoader(Subset(train_set, idx), batch_size=64)

activations = {}
gradients = {}

def fwd_hook(name):
    def hook(m,i,o): activations[name] = o.detach()
    return hook

def bwd_hook(name):
    def hook(m,gi,go): gradients[name] = go[0].detach()
    return hook

for name,m in model.named_modules():
    if isinstance(m, nn.Conv2d):
        m.register_forward_hook(fwd_hook(name))
        m.register_full_backward_hook(bwd_hook(name))

In [9]:
taylor = {}

criterion = nn.CrossEntropyLoss()

model.eval()

print("Calculating Importance...")

for x,y in tqdm(calib_loader):
    x,y = x.to(DEVICE), y.to(DEVICE)

    loss = criterion(model(x),y)

    model.zero_grad()
    loss.backward()

    for k in activations:
        score = torch.abs(activations[k]*gradients[k]).sum((2,3))
        taylor.setdefault(k,[]).append(score.cpu())

taylor_final = {k: torch.cat(v).mean(0) for k,v in taylor.items()}

print("✅ Importance computed")

Calculating Importance...


100%|██████████| 32/32 [00:04<00:00,  7.35it/s]

✅ Importance computed


In [10]:
def prune_structural(model, importance_dict):
    model = copy.deepcopy(model).to(DEVICE)
    model.eval()

    example_inputs = torch.randn(1, 3, 32, 32).to(DEVICE)

    DG = tp.DependencyGraph().build_dependency(
        model, example_inputs=example_inputs
    )

    for name, module in model.named_modules():

        # 🔥 Only prune deeper layers
        if name not in [
            "features.10","features.12","features.14",
            "features.17","features.19","features.21"
        ]:
            continue

        if isinstance(module, nn.Conv2d) and name in importance_dict:

            imp = importance_dict[name].to(DEVICE)

            # Normalize importance
            imp = imp / (imp.sum() + 1e-8)

            keep = int(module.out_channels * (1 - PRUNE_RATIO))
            prune = module.out_channels - keep

            if prune <= 0:
                continue

            idx = torch.argsort(imp)[:prune].tolist()

            group = DG.get_pruning_group(
                module,
                tp.prune_conv_out_channels,
                idxs=idx
            )

            if DG.check_pruning_group(group):
                group.prune()

    return model

In [11]:
pruned_model = prune_structural(model, taylor_final)

print("Before pruning:", evaluate(model)*100)
print("After pruning :", evaluate(pruned_model)*100)

torch.save(pruned_model.state_dict(),
           "/kaggle/working/vgg16_pruned_initial.pt")

Before pruning: 91.0
After pruning : 75.33


In [12]:
optimizer = torch.optim.SGD(
    pruned_model.parameters(),
    lr=LR,
    momentum=0.9,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_FINETUNE
)

criterion = nn.CrossEntropyLoss()

In [13]:
for param in pruned_model.features.parameters():
    param.requires_grad = False

for epoch in range(2):   # reduced
    pruned_model.train()
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(pruned_model(x),y)
        loss.backward()
        optimizer.step()

    print(f"Warmup Epoch {epoch+1} | Acc {evaluate(pruned_model)*100:.2f}%")

Warmup Epoch 1 | Acc 88.24%
Warmup Epoch 2 | Acc 88.38%


In [14]:
for param in pruned_model.parameters():
    param.requires_grad = True

In [15]:
best_acc = 0

print("Finetuning...")

for epoch in range(EPOCHS_FINETUNE):

    pruned_model.train()

    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(pruned_model(x),y)
        loss.backward()
        optimizer.step()

    scheduler.step()

    acc = evaluate(pruned_model)

    print(f"Epoch {epoch+1} | Acc {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(pruned_model.state_dict(),
                   "/kaggle/working/vgg16_CIFAR10_pruned_best.pt")
        print("Model Saved with Highest Accuracy : ",best_acc)

Finetuning...
Epoch 1 | Acc 87.90%
Model Saved with Highest Accuracy :  0.879
Epoch 2 | Acc 86.89%
Epoch 3 | Acc 88.76%
Model Saved with Highest Accuracy :  0.8876
Epoch 4 | Acc 87.87%
Epoch 5 | Acc 86.76%
Epoch 6 | Acc 89.28%
Model Saved with Highest Accuracy :  0.8928
Epoch 7 | Acc 86.51%
Epoch 8 | Acc 89.38%
Model Saved with Highest Accuracy :  0.8938
Epoch 9 | Acc 89.18%
Epoch 10 | Acc 88.63%
Epoch 11 | Acc 89.68%
Model Saved with Highest Accuracy :  0.8968
Epoch 12 | Acc 89.33%
Epoch 13 | Acc 89.89%
Model Saved with Highest Accuracy :  0.8989
Epoch 14 | Acc 89.43%
Epoch 15 | Acc 89.72%
Epoch 16 | Acc 89.42%
Epoch 17 | Acc 89.07%
Epoch 18 | Acc 89.84%
Epoch 19 | Acc 89.92%
Model Saved with Highest Accuracy :  0.8992
Epoch 20 | Acc 89.95%
Model Saved with Highest Accuracy :  0.8995
Epoch 21 | Acc 89.67%
Epoch 22 | Acc 89.58%
Epoch 23 | Acc 89.67%
Epoch 24 | Acc 88.81%
Epoch 25 | Acc 90.05%
Model Saved with Highest Accuracy :  0.9005
Epoch 26 | Acc 90.20%
Model Saved with Highest Acc

In [16]:
def evaluate_comprehensive(model, dataloader):
    model.eval()

    top1, top5, total = 0,0,0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x,y in dataloader:
            x,y = x.to(DEVICE), y.to(DEVICE)

            outputs = model(x)
            _, pred = outputs.topk(5,1,True,True)
            pred = pred.t()

            correct = pred.eq(y.view(1,-1).expand_as(pred))

            top1 += correct[0].sum().item()
            top5 += correct[:5].sum().item()
            total += y.size(0)

            all_preds.extend(pred[0].cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    top1 = top1/total*100
    top5 = top5/total*100

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )

    print("\n===== FINAL METRICS =====")
    print(f"Top-1 Accuracy : {top1:.2f}%")
    print(f"Top-5 Accuracy : {top5:.2f}%")
    print(f"Precision      : {precision*100:.2f}%")
    print(f"Recall         : {recall*100:.2f}%")
    print(f"F1 Score       : {f1*100:.2f}%")

evaluate_comprehensive(pruned_model, test_loader)


===== FINAL METRICS =====
Top-1 Accuracy : 92.12%
Top-5 Accuracy : 99.69%
Precision      : 92.10%
Recall         : 92.12%
F1 Score       : 92.10%
